# GraphEm RAPIDS on CUDA

This notebook exercises the current `graphem_rapids.GraphEmbedder` interface. It deliberately has no saved results: an executed notebook is a local inspection aid, not a benchmark or scientific record.

The cells require a working NVIDIA CUDA device and the CUDA packages listed in the repository's installation instructions. For the qualified reproduction setting, use Python 3.11, CUDA 12.9, Torch 2.11, CuPy 14.1.1, cuVS 26.06, and an NVIDIA H100 with 80 GB of memory. Another CUDA GPU may be useful for experimentation, but it does not reproduce that hardware contract. There is no CPU fallback.

In [ ]:
import numpy as np
import torch

import graphem_rapids as gr

if not torch.cuda.is_available():
    raise RuntimeError("This notebook requires a working CUDA device.")

cuda_device = torch.cuda.get_device_name(0)
cuda_device

## Build and run one fixed example

The graph, sampled edges, and iteration count are fixed here. The constructor also keeps midpoint queries at the canonical ceiling of 64.

In [ ]:
adjacency = gr.generate_er(n=1_000, p=0.1, seed=0)
embedder = gr.GraphEmbedder(
    adjacency=adjacency,
    n_components=3,
    L_min=40.0,
    k_attr=1.0,
    k_inter=1.0,
    n_neighbors=15,
    sample_size=2_048,
    midpoint_query_batch_size=64,
    seed=0,
    device="cuda",
    verbose=False,
)

In [ ]:
embedder.run_layout(num_iterations=30)

positions = embedder.get_positions()
scores = embedder.get_scores()
top_vertices = embedder.get_top_k(50)
diagnostics = embedder.get_diagnostics()

assert positions.shape == (1_000, 3)
assert scores.shape == (1_000,)
assert np.isfinite(positions).all()
assert np.isfinite(scores).all()

run_summary = {
    "cuda_device": cuda_device,
    "positions_shape": positions.shape,
    "score_range": (float(scores.min()), float(scores.max())),
    "top_vertices": top_vertices.tolist(),
    "algorithm": diagnostics["algorithm"],
    "iterations": diagnostics["iterations"],
}
run_summary

## Reading the result

The final dictionary summarizes this invocation. The diagnostic receipt records its configuration and algorithm identity. Neither is a sealed qualification artifact, and no timing, scaling, or scientific claim should be inferred from it.